# 08 — Three-legged OAuth consent

The agent is **out of the credential chain**.

Up to this point, the agent has been the trusted intermediary: it knew how
to ask Keycloak for user tokens (via direct grant in pattern 5+), or it
held a service credential (in patterns 1-3). Either way, if the agent
were compromised, an attacker could obtain user data without ever
involving the user.

In this pattern, the *user* goes through Keycloak's consent screen in
their own browser, the user authenticates to Keycloak directly, and
Keycloak issues an access token to a *different client* (`user-direct-client`,
not `agent-client`). The user pastes that token to the agent, and the
agent uses it to call the tool on the user's behalf. The agent never had
the password. The agent never spoke to Keycloak's authentication endpoint
on the user's behalf.

## Why this is different from every previous pattern

Every previous strategy (`fetch_user_jwt`, `exchange_token`) authenticated
to Keycloak using the *agent*'s confidential client credentials and the
user's password. The agent had to know the password. That's a model that
breaks down the moment the agent is exposed to untrusted data — a
prompt-injected agent could be tricked into using its credentials in
ways the user never approved.

3-legged OAuth flips this. The user holds their own credentials and
delegates a specific scoped capability to the agent via a consent
screen. The agent receives only what the user explicitly authorized.

```
   user                Keycloak                    agent
    |                     |                          |
    |---password------>   |                          |
    |   (in browser)      |                          |
    |                     |--consent screen-->user   |
    |                     |<-yes-------------------user
    |                     |--access token---------> |
    |                                                |--Bearer token-->expense-service
```

The agent has no idea what the user's password is. The agent has only
a token that the user explicitly granted, with whatever scope was on the
consent screen.

## This notebook is interactive

`run-all` won't work for this notebook — it requires you to open a URL
in your browser, log in as one of the demo users, click "consent",
and paste the resulting `code` value back into the notebook. Take it
cell by cell.

You will see a "this site can't be reached" error in your browser after
consenting. **That's expected** — there is no callback server. The
consent flow puts the authorization code in the URL bar before the
browser tries to load the (nonexistent) callback page. You read the
`code=...` query parameter directly out of the URL bar and paste it back.

In [ ]:
from shared.display import three_legged_login, show_token, run_as, show_what_tool_saw
from shared.auth import ThreeLeggedOAuthAuth
from shared.agent import Agent
from shared.tools import ALL_TOOLS
from shared.config import EXPENSE_SERVICE_URL

## Step 1: walk through the consent flow

Run the cell below. It will print a URL. Open it in your browser.

You will see Keycloak's login screen. Log in as `alice` (password
`password`). After login, Keycloak will show a consent screen asking
whether to grant `user-direct-client` permission to act on alice's
behalf. Click "yes".

Your browser will then try to redirect to `http://localhost:8765/callback`,
which doesn't exist. The page will fail to load. **Look at the URL bar**.
You'll see something like:

    http://localhost:8765/callback?state=xxx&session_state=xxx&code=ABCD-EFGH-...

Copy just the `code` value (the part after `code=`, before any `&` if
present), and paste it into the input prompt below.

In [ ]:
access_token = three_legged_login(scope="openid")

## Step 2: inspect the token

The interesting claim is `azp` — it identifies which client this token
was issued to. In every previous pattern, every token had `azp =
agent-client`. This one has `azp = user-direct-client`. The agent never
asked Keycloak for this token; the user did.

In [ ]:
show_token(access_token, label="user-direct-client access token (the agent never had this user's password)")

## Step 3: hand the token to an agent and run a real prompt

The `ThreeLeggedOAuthAuth` strategy is constructed with the token already
in hand. The agent has no way to obtain a *different* token — it can only
use what was granted to it.

In [ ]:
strategy = ThreeLeggedOAuthAuth(access_token=access_token)
agent = Agent(strategy=strategy, tools=ALL_TOOLS)

result = run_as("alice", "What expenses do I have?", agent)
show_what_tool_saw(EXPENSE_SERVICE_URL)

## Tradeoffs

- **Where authz lives:** the same place as in pattern 5/6 (the tool, via
  validated JWT) — what changes is **how the token gets to the agent**.
- **The agent is no longer in the trust chain for credentials.** The agent
  gets a token; the agent cannot get a *different* token without going
  through the user again. Compromise of the agent gives the attacker
  access only to what the user already explicitly consented to, for the
  duration of the token's validity.
- **User-revocable:** the user (or an admin) can revoke the
  `user-direct-client` session at any time via Keycloak's user account
  console, and the agent's calls will start failing.
- **Operationally heavier:** every "session" the user has with the agent
  needs a consent flow. Not appropriate for high-frequency interactions —
  this is the pattern for *granting an agent narrow long-lived access to
  do specific things*, not for every-API-call agent loops.
- **The right pattern when:** you don't trust the agent's runtime,
  the agent runs on infrastructure the user doesn't control, the user
  needs to be able to revoke at any time, or compliance demands explicit
  user consent for each capability the agent uses.
- **The wrong pattern when:** the agent runs in the user's own trusted
  runtime (e.g., a CLI tool the user installed), high-frequency
  interactions are expected, or user friction is unacceptable.

Note that nothing prevents you from *combining* this with token exchange:
the user-issued token can itself be exchanged for a narrowed token before
calling each tool, giving you both "user explicitly consented" AND
"narrowed audience per call". Production systems with high security
requirements often do exactly that.

## What's NOT in this repo

This series covered eight patterns at the *identity and authz* dimension
of agent security. There are other dimensions worth knowing about that
this repo deliberately doesn't try to teach — see the README's "Out of
scope" section for pointers. Briefly:

- **Capability tokens (Biscuit, macaroons):** an alternative to bearer
  JWTs where authority is encoded as attenuable, offline-verifiable
  claims. Theoretically powerful but with limited adoption in agent
  systems as of 2026.
- **mTLS / SPIFFE/SPIRE:** transport-layer mutual identity. Sits beneath
  every pattern in this repo. Painful to run locally for a teaching
  artifact, so we used cleartext HTTP and assumed the network boundary
  is trusted.
- **Gateway-mediated identity injection:** an API gateway terminates the
  user's auth at the edge and injects a verified identity header
  downstream to internal services. Common in service meshes; covered in
  the "Eight Dimensions" blog post.

That's it. Run-all on every notebook in order and the data should tell
the story by itself.